In [14]:
import asyncio
import nodriver as uc
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time
from unidecode import unidecode
import numpy as np
from itertools import chain

#since roster table is hidden in comments, needs this to fish it out
def makeCommentTable(soup1, type):
    #finds all comments in soup and makes them not comments and just html. 
    comments = soup1.find_all(string=lambda text: isinstance(text, Comment) and "table_container" in text)
    html = str(comments).replace('<!--', '').replace('-->', '')
    
    #parse through the HTML content using Beautiful Soup
    soup = BeautifulSoup(html, 'html.parser')

    #reads roster table html into a dataframe
    if type == False:
        #find the roster table by its HTML id
        rosterTable = soup.find('table', {'id': 'roster'})
        if rosterTable is None:
            rosterTable = soup1.find('table', {'id': 'roster'})
        df = pd.read_html(str(rosterTable))[0]
    else:
        passingTable = soup.find('table', {'id': 'passing'})
        otherTable = soup.find('table', {'id': 'rushing_and_receiving'})
        df = [pd.read_html(str(passingTable)), pd.read_html(str(otherTable))]
        df = list(chain.from_iterable(df))

    return df

async def fetch_page(browser, url, max_retries=5):
    """Fetch a page with retries. Waits for Cloudflare to resolve."""
    for attempt in range(max_retries):
        try:
            page = await browser.get(url)
            
            # Wait and check if we got past Cloudflare
            for wait in range(6):
                await asyncio.sleep(3)
                html = await page.evaluate('document.documentElement.outerHTML')
                if 'Just a moment' not in html and len(html) > 50000:
                    return html
            
            # If still on Cloudflare page after waiting, retry
            if 'Just a moment' in html or len(html) < 50000:
                print(f"  Attempt {attempt+1}: still on Cloudflare page, retrying...")
                await asyncio.sleep(10)
                continue
                
            return html
        except Exception as e:
            print(f"  Attempt {attempt+1} error: {e}")
            await asyncio.sleep(10)
    
    raise Exception(f"Failed to load {url} after {max_retries} attempts")

async def rosterMaker():

    #empty df to add things to
    df = pd.DataFrame(columns= ["No.", "Player", "Age", "Pos", "G", "GS", "Wt", "Ht", "College/Univ", "BirthDate", "Yrs", "AV", "Drafted (tm/rnd/yr)", "Year", "YearsBack", "Team"])

    #years used for grading. changes each season.
    years = ["2013","2014","2015","2016","2017","2018","2019","2020","2021","2022","2023"]
    #team abbr list
    teams = ["crd", "atl", "rav", "buf", "car", "chi", "cin", "cle", "dal", "den", "det", "gnb", "htx", "clt", "jax", "kan", "rai", "sdg", "ram", "mia", "min", "nwe", "nor", "nyg", "nyj", "phi", "pit", "sfo", "sea", "tam", "oti", "was"]
    x = 1

    # Launch browser once
    browser = await uc.start()
    
    # First load: visit homepage to establish Cloudflare cookies
    print("Establishing Cloudflare session...")
    first_page = await browser.get("https://www.pro-football-reference.com/")
    for _ in range(10):
        await asyncio.sleep(3)
        html = await first_page.evaluate('document.documentElement.outerHTML')
        if 'Just a moment' not in html:
            print("Cloudflare session established!")
            break
    else:
        print("Warning: Cloudflare may not have resolved yet")

    for num in years:
        for item in teams: 
            url = "https://www.pro-football-reference.com/teams/" + item + "/" + num + "_roster.htm"

            #rate limit: wait between requests
            await asyncio.sleep(5)

            html = await fetch_page(browser, url)
            print(f"{item} {num} - loaded ({len(html)} chars)")

            soup = BeautifulSoup(html, 'html.parser')
            table = makeCommentTable(soup, False)

            table['Year'] = num
            table["YearsBack"] = x
            table["Team"] = item

            df = pd.concat([df, table], ignore_index=True, join="inner")
                
        x+=1

    browser.stop()
    
    if len(years)==1:
        rbValues = ['RB', 'HB', 'TB', 'FB', "LH", "RH", "BB", "B", "WB"]
        df.loc[df['Pos'].isin(rbValues), 'Pos'] = "RB"
        df.loc[df['Pos'].str.len() > 2, 'Pos'] = df.loc[df['Pos'].str.len() > 2, 'Pos'].apply(lambda x: x[:2])
        keepers = ["RB", "QB", "TE", "WR"]
        df = df.loc[df['Pos'].isin(keepers)]
        df.to_pickle("PickleFiles/currYearRoster.pkl")
    else:
        df.to_pickle("PickleFiles/teamsPastRoster.pkl")

await rosterMaker()

CancelledError: 

In [ ]:
import pandas as pd
teamsPast = pd.read_pickle("PickleFiles/teamsPastRoster.pkl")
teamsPast['YearsBack'] = 2024 - teamsPast['Year'].astype(int)
teamsPast.to_pickle('PickleFiles/teamsPastRoster.pkl')

In [20]:
from pathlib import Path
import pandas as pd
import nfl_data_py as nfl

# Notebook is in: <project_root>/Models/Roster Creation/
# So project root is two levels up.
PROJECT_ROOT = Path.cwd().parents[1]

# Save into the *existing* Flask folder: <project_root>/webapp/data
WEBAPP_DATA_DIR = PROJECT_ROOT / "webapp" / "data"
WEBAPP_DATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = WEBAPP_DATA_DIR / "teamsPastRoster.pkl"

players = nfl.import_players()

cols = [
    "display_name", "first_name", "last_name",
    "latest_team", "position", "position_group",
    "height", "weight",
    "birth_date", "college_name", "college_conference",
    "rookie_season", "last_season", "status", "years_of_experience",
    "jersey_number",
    "draft_year", "draft_round", "draft_pick", "draft_team",
    "gsis_id", "pfr_id", "espn_id", "nfl_id",
]

df = players[cols].copy()

df = df.rename(columns={
    "display_name": "Player",
    "latest_team": "Team",
    "position": "Pos",
    "position_group": "PosGroup",
    "birth_date": "BirthDate",
    "college_name": "College",
    "college_conference": "CollegeConf",
    "rookie_season": "RookieSeason",
    "last_season": "LastSeason",
    "years_of_experience": "ExperienceYears",
    "jersey_number": "Jersey",
    "draft_year": "DraftYear",
    "draft_round": "DraftRound",
    "draft_pick": "DraftPick",
    "draft_team": "DraftTeam",
})

# Normalize status
df["status"] = df["status"].astype(str).str.strip().str.upper()

# 1) Keep "active" status rows (per nflverse status)
df = df[df["status"] == "ACT"]

# 2) Remove older entries that aren't relevant anymore
#    Choose the cutoff year you want. This means "last played season >= cutoff".
CUTOFF_LAST_SEASON = 2024
df["LastSeason"] = pd.to_numeric(df["LastSeason"], errors="coerce")
df = df[df["LastSeason"].fillna(0).astype(int) >= CUTOFF_LAST_SEASON]

# Keep fantasy positions
df = df.dropna(subset=["Player", "Team", "Pos"])
df = df[df["Pos"].isin({"QB", "RB", "WR", "TE"})]

# Put status FIRST so you can confirm easily
df = df[["status"] + [c for c in df.columns if c != "status"]]

# Nice ordering
df = df.sort_values(["Team", "Pos", "Player"]).reset_index(drop=True)

print("Status counts after filters:")
print(df["status"].value_counts(dropna=False))

print("\nLastSeason min/max after filters:")
print(int(df["LastSeason"].min()), int(df["LastSeason"].max()))

# Save for Flask
df.to_pickle(OUT_PATH)

print("\nProject root:", PROJECT_ROOT)
print("Saved ->", str(OUT_PATH))
print("Rows:", len(df))

# Preview (status first)
df[["status", "Player", "Pos", "Team", "LastSeason", "RookieSeason", "Jersey", "height", "weight"]].head(25)

Status counts after filters:
status
ACT    522
Name: count, dtype: int64

LastSeason min/max after filters:
2024 2025

Project root: c:\Users\deven\StepToolKit\StepByStepToolKit
Saved -> c:\Users\deven\StepToolKit\StepByStepToolKit\webapp\data\teamsPastRoster.pkl
Rows: 522


,status,Player,Pos,Team,LastSeason,RookieSeason,Jersey,height,weight
0,ACT,Jacoby Brissett,QB,ARI,2025,2016,7,76.0,235.0
1,ACT,Kedon Slovis,QB,ARI,2025,2024,19,74.0,223.0
2,ACT,Corey Kiner,RB,ARI,2025,2025,None,69.0,210.0
3,ACT,Emari Demercado,RB,ARI,2025,2023,31,69.0,215.0
4,ACT,Michael Carter,RB,ARI,2025,2021,22,68.0,201.0
5,ACT,Elijah Higgins,TE,ARI,2025,2023,84,75.0,245.0
6,ACT,Josiah Deguara,TE,ARI,2025,2020,47,74.0,240.0
7,ACT,Pharaoh Brown,TE,ARI,2025,2017,None,77.0,246.0
8,ACT,Rivaldo Fairweather,TE,ARI,2025,2025,None,75.0,249.0
9,ACT,Trey McBride,TE,ARI,2025,2022,85,76.0,246.0


In [15]:
players["status"].value_counts(dropna=False)

status
ACT    13717
RES     3523
CUT     3410
DEV     3138
RSN      157
PUP      127
NWT      121
RSR       82
SUS       39
RET       33
EXE        6
INA        3
Name: count, dtype: int64